# Post-processing sweep for a crop U-Net, ResNet-34 by default (no retraining)

Import this notebook into Kaggle and press **Save & Run All**. None of the cells asks for input. With the default `MODEL = "r34"` no cell needs editing. To sweep another model, change only `MODEL` in the settings cell and attach that model's training output:

| `MODEL` | Encoder | Save the training notebook as | Checkpoint |
| --- | --- | --- | --- |
| `r34` (default) | `resnet34` | `train-unet-r34-crops-80ep` | `unet_r34_crops.pt` |
| `r50` | `resnet50` | `train-unet-r50-crops` | `unet_r50_crops.pt` |
| `effb3` | `efficientnet-b3` | `train-unet-effb3-crops` | `unet_effb3_crops.pt` |
| `r34_v2` | `resnet34` | `train-unet-r34-v2` | `unet_r34_v2.pt` |
| `r34_v2_agree` | `resnet34` | `train-unet-r34-v2-agree` | `unet_r34_v2_agree.pt` |

It loads the checkpoint written by `train-unet-r34-crops-80ep`, predicts each of the 142 holdout frames **once** (tiled, native 2048 px), and scores a grid of post-processing settings on those cached probability maps: probability **threshold**, closing kernel **`close_size`** and **`min_area`**. It writes `/kaggle/working/holdout_postproc_<MODEL>.csv`, sorted by pooled per-view PQ, best first. It does not train, does not read the test images, does not write `submission.csv` and does not submit to Kaggle.

**Attach exactly these Kaggle inputs**

1. The competition `filament-segmentation-2026` (train images and train JSON).
2. The dataset `ignacioaguirrecandia/solar-code` (the repo zip, extracted as `Solar-Filament-Segmentation-main/`), a version that contains `filament.checkpoint.CROP_CHECKPOINTS`. [VERIFY] re-upload it after this PR is merged; an older version stops at the imports.
3. The **output of the training notebook of `MODEL`**: by default `train-unet-r34-crops-80ep` (Add Input → Notebook Output). The default `WEIGHTS_PATH` is `/kaggle/input/notebooks/ignacioaguirrecandia/train-unet-r34-crops-80ep/unet_r34_crops.pt`.

**Settings**

- Accelerator: **GPU T4**, a single T4.
- Internet: **On**, for the `pip install` of `segmentation-models-pytorch==0.6.0`. The checkpoint holds the encoder, so no encoder weights are downloaded.

## 1. `src` on the path

`src` is the directory that contains `filament/crop_training.py`. A checkout in the working directory is used as it is. Otherwise the walk goes through `/kaggle/input` at any depth, skips every `competitions` directory, and takes the shallowest match. It is used in place and not copied.

In [ ]:
import os
import sys
from pathlib import Path


def _local_src() -> Path | None:
    """A checkout that already has the crop modules (a run off Kaggle)."""
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        src = candidate / "src"
        if (src / "filament" / "crop_training.py").is_file():
            return src
    return None


def _private_dataset_src(root: Path | None = None) -> Path | None:
    """Return the shallowest private-dataset ``src`` under ``root``.

    ``filament/crop_training.py`` may sit at any depth. Directories named
    ``competitions`` are not walked, so the MAGFiLO tree is left alone.
    """
    root = Path("/kaggle/input") if root is None else root
    if not root.is_dir():
        return None
    marker = Path("filament") / "crop_training.py"
    matches: list[Path] = []
    for dirpath, dirnames, _filenames in os.walk(root, topdown=True):
        dirnames[:] = [name for name in dirnames if name != "competitions"]
        current = Path(dirpath)
        if current.name != "src" or not (current / marker).is_file():
            continue
        matches.append(current)
        dirnames.clear()
    if not matches:
        return None
    return min(
        matches,
        key=lambda path: (len(path.relative_to(root).parts), path.as_posix()),
    )


def _put_on_path(src: Path) -> None:
    text = str(src)
    if text not in sys.path:
        sys.path.insert(0, text)
    current = os.environ.get("PYTHONPATH", "")
    parts = [part for part in current.split(os.pathsep) if part]
    if text not in parts:
        os.environ["PYTHONPATH"] = os.pathsep.join([text, *parts]) if parts else text


def _prepare_src() -> Path:
    """Put the repo's ``src`` on ``sys.path`` and return it.

    A checkout in the working directory is used as it is. Otherwise the
    private dataset's ``src`` is used in place. It is not copied to
    ``/kaggle/working``: a copy there would be saved with this notebook's
    output and found again by a later notebook that attaches that output.
    """
    local = _local_src()
    if local is not None:
        _put_on_path(local)
        return local
    source = _private_dataset_src()
    if source is None:
        raise FileNotFoundError(
            "Could not find src/filament/crop_training.py in the working "
            "directory or under /kaggle/input (any depth outside "
            "competitions). Attach the dataset ignacioaguirrecandia/solar-code, "
            "for example "
            "/kaggle/input/datasets/ignacioaguirrecandia/solar-code/"
            "Solar-Filament-Segmentation-main/src. If it is attached, its "
            "version predates the crop modules: upload a new version of the "
            "dataset from the branch that has them. Do not make the GitHub "
            "repository public."
        )
    _put_on_path(source)
    return source


SRC = _prepare_src()
print(f"src={SRC}")
PYTHON = sys.executable

## 2. Install the segmentation library

`segmentation-models-pytorch` **0.6.0** (MIT), the version that trained the checkpoint. Kaggle already has PyTorch; this cell does not install it.

In [ ]:
import subprocess

# Pinned. segmentation-models-pytorch 0.6.0 is MIT.
SMP_VERSION = "0.6.0"

completed = subprocess.run(
    [
        PYTHON,
        "-m",
        "pip",
        "install",
        f"segmentation-models-pytorch=={SMP_VERSION}",
    ],
    check=False,
)
if completed.returncode != 0:
    raise RuntimeError(
        f"segmentation-models-pytorch=={SMP_VERSION} did not install. Stop. "
        "Internet has to be On. Kaggle already has PyTorch; this cell only "
        "installs segmentation-models-pytorch (MIT)."
    )

In [ ]:
import segmentation_models_pytorch as smp

print(f"segmentation-models-pytorch {smp.__version__}")
if smp.__version__ != SMP_VERSION:
    raise RuntimeError(
        f"Expected segmentation-models-pytorch {SMP_VERSION}, "
        f"found {smp.__version__}. Stop."
    )

## 3. Settings and where the weights are

The grid is 7 thresholds × 4 `close_size` × 6 `min_area` = 168 settings. The current setting (threshold 0.5, `close_size` 0, `min_area` 400) is in it, so its row can be compared with the training notebook's sweep (`holdout_pq` of the checkpoint entry: 0.3831 for `r34`, `docs/RESULTS.md`).

- `THRESHOLDS`: probability cut, `probability >= threshold`.
- `CLOSE_SIZES`: side of the elliptical closing kernel in pixels. `0` is no closing and the current value. It cannot go lower, so the grid goes up through odd sizes: 3 and 7 join gaps of a few pixels, 15 joins fragments about a dozen pixels apart at the native resolution.
- `MIN_AREAS`: components smaller than this many pixels (on the 2048 × 2048 frame) are dropped.

`MODEL` is a name of `filament.checkpoint.CROP_CHECKPOINTS`. `WEIGHTS_PATH` takes its default from that entry's `weights_path`. Set it to another `.pt` file to load that one, or to `None` to search every `*.pt` under `/kaggle/working` and `/kaggle/input` (skipping `competitions`). The helpers that find the file, and re-zip a Kaggle-extracted archive, are the ones of `notebooks/infer_unet_crops.ipynb`.

**Checkpoint guard.** The stored `epochs`, the encoder and, once pinned, `tensor_sha256` must match the entry of `MODEL`; otherwise the notebook stops (`filament.checkpoint`). `ALLOW_UNVERIFIED_WEIGHTS = True` turns a mismatch into a warning; leave it `False`.

`TILE_OVERLAP` (128), `INFER_BATCH_SIZE` (8) and `USE_AMP` are the training notebook's values, so the probabilities are the ones its holdout sweep used.

In [ ]:
import zipfile

from filament.checkpoint import crop_checkpoint

# ---- Which model: a name of filament.checkpoint.CROP_CHECKPOINTS --------------
MODEL = "r34"
EXPECTED = crop_checkpoint(MODEL)

# ---- Post-processing grid ----------------------------------------------------
THRESHOLDS = (0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60)
CLOSE_SIZES = (0, 3, 7, 15)
MIN_AREAS = (240, 320, 400, 500, 600, 700)
# The setting in use today; its row is printed as a cross-check.
CURRENT = (0.5, 0, 400)

# ---- Weights and the checkpoint guard ---------------------------------------
# Default from the checkpoint config. Another .pt path loads that file; None
# searches /kaggle/working and /kaggle/input (skipping competitions).
WEIGHTS_PATH = EXPECTED.weights_path
EXPECTED_EPOCHS = EXPECTED.epochs
EXPECTED_TENSOR_SHA256 = EXPECTED.tensor_sha256
ALLOW_UNVERIFIED_WEIGHTS = False

# ---- Tiled inference, as in the training notebook's holdout evaluation -------
TILE_OVERLAP = 128
INFER_BATCH_SIZE = 8
USE_AMP = True
SEED = 0

OUTPUT_PATH = Path(f"/kaggle/working/holdout_postproc_{MODEL}.csv")

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")
PREFERRED_NAME = Path(EXPECTED.weights_path).name


def _pt_under(root: Path) -> list[Path]:
    """Every ``*.pt`` file or directory under ``root``.

    A directory counts because Kaggle extracts an uploaded ``.pt`` zip
    into a folder of that name. ``competitions`` is not walked.
    """
    found: list[Path] = []
    if not root.is_dir():
        return found
    for dirpath, dirnames, filenames in os.walk(root):
        directory = Path(dirpath)
        kept: list[str] = []
        for name in sorted(dirnames):
            if name == "competitions":
                continue
            if name.endswith(".pt"):
                found.append(directory / name)
                continue
            kept.append(name)
        dirnames[:] = kept
        for name in sorted(filenames):
            if name.endswith(".pt"):
                found.append(directory / name)
    return found


def _weight_candidates() -> list[Path]:
    """``.pt`` files and extracted archives, working then input."""
    return _pt_under(KAGGLE_WORKING) + _pt_under(KAGGLE_INPUT)


def _archive_error(path: Path, reason: str) -> RuntimeError:
    """Explain a Kaggle-extracted ``.pt`` that cannot be re-zipped."""
    return RuntimeError(
        f"{path} is a directory, not a checkpoint file. "
        "Kaggle extracts an uploaded .pt because torch.save writes a zip, "
        "so the file becomes a folder such as unet.pt/unet/data.pkl. "
        f"This notebook did not re-zip it: {reason} "
        "Set WEIGHTS_PATH to a real .pt file that Kaggle did not extract."
    )


def _rezip_extracted_archive(directory: Path, dest: Path | None = None) -> Path:
    """Re-zip one extracted torch archive to a file under working.

    Members keep the archive folder name (``unet/data.pkl``). The zip is
    stored, not deflated, so ``torch.load`` can read it. The directory
    itself is never returned.
    """
    if not directory.is_dir():
        raise _archive_error(directory, "the path is not a directory.")
    pkls = sorted(
        path
        for path in directory.rglob("data.pkl")
        if path.is_file() and not path.is_symlink()
    )
    if len(pkls) != 1:
        raise _archive_error(
            directory,
            f"expected one data.pkl, found {len(pkls)}.",
        )
    archive_dir = pkls[0].parent
    if archive_dir.parent == archive_dir:
        raise _archive_error(directory, "data.pkl has no parent folder.")
    names = {child.name for child in archive_dir.iterdir()}
    storage = (archive_dir / "data").is_dir()
    marked = "version" in names or "byteorder" in names or storage
    if "data.pkl" not in names or not marked:
        raise _archive_error(
            directory,
            "data.pkl is not next to version, byteorder, or a data folder.",
        )
    files: list[Path] = []
    ordered = sorted(archive_dir.rglob("*"), key=lambda item: item.as_posix())
    for path in ordered:
        if path.is_symlink():
            raise _archive_error(directory, f"{path.name} is a symlink.")
        if path.is_file():
            files.append(path)
    if not files:
        raise _archive_error(directory, "the archive folder has no files.")
    try:
        KAGGLE_WORKING.mkdir(parents=True, exist_ok=True)
    except OSError as exc:
        raise _archive_error(
            directory,
            f"/kaggle/working is not writable ({exc}).",
        ) from exc
    stem = directory.stem if directory.name.endswith(".pt") else directory.name
    if dest is None:
        dest = KAGGLE_WORKING / f"{stem}-from-extracted.pt"
    try:
        dest.resolve().relative_to(archive_dir.resolve())
    except ValueError:
        inside = False
    else:
        inside = True
    if inside:
        raise _archive_error(
            directory,
            "the re-zipped file would land inside the extracted archive.",
        )
    with zipfile.ZipFile(dest, "w", compression=zipfile.ZIP_STORED) as handle:
        for file in files:
            rel = file.relative_to(archive_dir.parent).as_posix()
            if rel.startswith("/") or ".." in rel.split("/"):
                raise _archive_error(directory, f"unsafe member {rel}.")
            handle.write(file, arcname=rel)
    print(f"Kaggle extracted the torch archive at {directory}. Re-zipped it to {dest}.")
    return dest


def _materialize_checkpoint(path: Path) -> Path:
    """Return a real ``.pt`` file. Re-zip an extracted archive."""
    if path.is_file():
        return path
    if path.is_dir():
        return _rezip_extracted_archive(path)
    raise FileNotFoundError(
        f"{path} does not exist. Set WEIGHTS_PATH to an existing .pt file."
    )


def _resolve_weights(explicit):
    """Return the checkpoint file to load.

    ``explicit`` that exists is used (an extracted archive is re-zipped). The
    default path missing, or ``None``, searches ``/kaggle/working`` and
    ``/kaggle/input``: a file or folder named ``PREFERRED_NAME`` first, then
    a lone ``.pt``. A path that was set by hand and is missing is an error.
    """
    if explicit is not None:
        path = Path(explicit)
        if path.exists():
            return _materialize_checkpoint(path)
        if str(explicit) != EXPECTED.weights_path:
            raise FileNotFoundError(
                f"WEIGHTS_PATH is set to {path}, but that file does not exist."
            )
        print(f"{path} does not exist. Searching for {PREFERRED_NAME}.")
    candidates = _weight_candidates()
    named = [path for path in candidates if path.name == PREFERRED_NAME]
    if len(named) == 1:
        return _materialize_checkpoint(named[0])
    if not named and len(candidates) == 1:
        print(f"No {PREFERRED_NAME}; using the only .pt found: {candidates[0]}")
        return _materialize_checkpoint(candidates[0])
    listed = "\n".join(f"  {path}" for path in candidates) or "  (none)"
    detail = (
        f"Several files named {PREFERRED_NAME} were found."
        if named
        else f"No {PREFERRED_NAME} and not exactly one .pt were found."
    )
    raise RuntimeError(
        f"{detail} Attach the output of the notebook "
        f"{Path(EXPECTED.weights_path).parent.name}, or "
        f"set WEIGHTS_PATH to the file to load.\n{listed}"
    )

## 4. Load the checkpoint and check it

Prints the stored fields and `tensor_sha256`, runs the guard, and rebuilds the network from the checkpoint with `encoder_weights=None`. The cell stops if there is no CUDA GPU.

In [ ]:
import random

import numpy as np
import torch

from filament import crop_training as ct
from filament.checkpoint import checkpoint_fingerprint, verify_and_report

if not hasattr(ct, "run_postproc_grid"):
    raise RuntimeError(
        "The attached solar-code dataset predates the post-processing grid "
        "(filament.crop_training.run_postproc_grid). Upload a new version of "
        "the dataset from a checkout that has it."
    )

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = ct.require_cuda(6.0)
print(f"device={torch.cuda.get_device_name(0)} torch={torch.__version__}")

WEIGHTS_FILE = _resolve_weights(WEIGHTS_PATH)
print(f"loaded weights={WEIGHTS_FILE}")
checkpoint = ct.load_checkpoint(WEIGHTS_FILE)
fingerprint = checkpoint_fingerprint(checkpoint)
for key in (
    "epochs",
    "status",
    "encoder_name",
    "crop_size",
    "tile_size",
    "smp_version",
):
    print(f"  {key}={checkpoint.get(key, 'not stored')}")
print(f"  tensor_sha256={fingerprint['tensor_sha256']}")

# Raises CheckpointMismatchError on the wrong weights unless
# ALLOW_UNVERIFIED_WEIGHTS is True.
verify_and_report(
    fingerprint,
    allow_unverified=ALLOW_UNVERIFIED_WEIGHTS,
    expected=EXPECTED,
    expected_epochs=EXPECTED_EPOCHS,
    expected_tensor_sha256=EXPECTED_TENSOR_SHA256,
)

TILE_SIZE = int(checkpoint.get("tile_size") or checkpoint["crop_size"])
model = ct.model_from_checkpoint(checkpoint, device)
del checkpoint
print(
    f"tile_size={TILE_SIZE} overlap={TILE_OVERLAP} batch={INFER_BATCH_SIZE} amp={USE_AMP}"
)

## 5. Date holdout

The same split as the training notebook: `filament.train_data.split_training_images` holds out the most recent 20% of calendar dates (142 files with 231 annotator views on the current data). Only the train images and the train JSON are read.

In [ ]:
from filament.annotations import (
    TRAIN_ANNOTATIONS_PATH,
    TRAIN_IMAGES_DIR,
    load_train_annotations,
)
from filament.sweep import count_views
from filament.train_data import split_training_images

IMAGE_DIR = Path(TRAIN_IMAGES_DIR)
ANNOTATIONS_PATH = Path(TRAIN_ANNOTATIONS_PATH)
if not ANNOTATIONS_PATH.is_file() or not IMAGE_DIR.is_dir():
    raise FileNotFoundError(
        "Train data was not found. Checked "
        f"images={IMAGE_DIR} and annotations={ANNOTATIONS_PATH}. "
        "Attach the competition filament-segmentation-2026 to this notebook."
    )
loaded = load_train_annotations(ANNOTATIONS_PATH)
split = split_training_images(loaded.images)
holdout_views = count_views(split.holdout)
print(
    f"train files={len(split.train)} holdout files={len(split.holdout)} "
    f"holdout annotator views={holdout_views}"
)
if (len(split.train), len(split.holdout), holdout_views) != (565, 142, 231):
    print(
        "WARNING: the split is not 565 / 142 files with 231 views, the numbers "
        "the training notebook used. [VERIFY] the data version."
    )
print("test/test_images is not used")

## 6. Predict once, score the grid

Each holdout frame is read at 2048 px and passed through the network **once** (overlapped tiles, probabilities averaged). For every threshold the mask is `probability >= threshold`; for every `close_size` the components are labeled once and every `min_area` is cut from them (`filament.instances.labels_for_min_areas`, identical to `labels_from_semantic`). The score is the pooled per-view PQ of `filament.sweep` (`fast_view_counts`, `PqCounts` added over all views, one `.pq` at the end); the metric is not reimplemented.

The network runs once per frame, but the scoring runs 168 times per annotator view. Its duration was not measured [VERIFY]; the progress line prints every 10 frames.

One holdout is one sample of dates, and 168 settings are compared on it. A gain of a few thousandths of PQ over the current setting is within what this selection can produce on its own and is not evidence of a better setting. Change the defaults only for a clear, broad gain.

In [ ]:
grid = ct.run_postproc_grid(
    model,
    split.holdout,
    IMAGE_DIR,
    device,
    thresholds=THRESHOLDS,
    close_sizes=CLOSE_SIZES,
    min_areas=MIN_AREAS,
    out_csv=OUTPUT_PATH,
    tile_size=TILE_SIZE,
    overlap=TILE_OVERLAP,
    infer_batch_size=INFER_BATCH_SIZE,
    use_amp=USE_AMP,
    predictor_name=Path(EXPECTED.weights_path).stem,
)
print(ct.describe_postproc_grid(grid, OUTPUT_PATH))

## 7. Compare with the current setting

The row for the current setting (threshold 0.5, `close_size` 0, `min_area` 400) is looked up in the grid and printed with its rank. The training notebook's sweep gave the checkpoint entry's `holdout_pq` for it (0.3831 for the 80-epoch `r34`, `docs/RESULTS.md`; `None` until a run is recorded, and then there is no comparison); a value that differs by more than about 0.005 means these are not the same weights, data or tiling, and the grid should not be trusted until that is explained.

In [ ]:
cut, kernel, area = CURRENT
current_rows = [
    (rank, row)
    for rank, row in enumerate(grid.rows, start=1)
    if (float(row["threshold"]), int(row["close_size"]), int(row["min_area"]))
    == (cut, kernel, area)
]
if not current_rows:
    print(f"The current setting {CURRENT} is not in the grid.")
else:
    rank, row = current_rows[0]
    print(
        f"current setting {CURRENT}: pooled_pq={row['pooled_pq']} rank {rank} of {len(grid.rows)}"
    )
    print(f"best setting:   {grid.rows[0]}")
    if EXPECTED.holdout_pq is None:
        print(f"No holdout_pq is recorded for {MODEL!r}, so there is no cross-check.")
    elif abs(float(row["pooled_pq"]) - EXPECTED.holdout_pq) > 0.005:
        print(
            "WARNING: the current setting differs from the "
            f"{EXPECTED.holdout_pq} of the training notebook."
        )
print(
    f"inference {grid.inference_seconds:.0f} s, "
    f"inference + labeling + scoring {grid.total_seconds:.0f} s"
)
print(f"Output: {OUTPUT_PATH.name} in /kaggle/working.")
print("This notebook wrote no submission file and did not submit to Kaggle.")